In [1]:
!pip install requests
!pip install beautifulsoup4

In [ ]:
import json
import re
import shutil
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

# --------------------------------------------
# Configuration
# --------------------------------------------

BASE_URL = "https://www.vanderbilt.edu/physicsdemonstration/davesdemos/demonstrations/"
START = 1
END = 206

OUTPUT_FILE = Path("../src/data/demos-scraped.json")
IMAGE_DIR = Path("../public/demo-images")

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

if IMAGE_DIR.exists():
    shutil.rmtree(IMAGE_DIR)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# --------------------------------------------
# Helpers
# --------------------------------------------

def clean_text(text):
    text = re.sub(r"Copyright © \d{4}, Vanderbilt University\. All Rights Reserved\.", "", text)
    text = re.sub(r"Writeup created by [^.]+\.", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_filename(url):
    parsed = urlparse(url)
    return Path(parsed.path).name


def download_image(img_url, save_dir, filename=None):
    try:
        response = requests.get(img_url, headers=HEADERS, timeout=15)

        if response.status_code != 200:
            print(f"  Failed image: {img_url}")
            return None

        if filename is None:
            filename = safe_filename(img_url)
        if not filename:
            filename = "image.jpg"

        save_path = save_dir / filename
        with open(save_path, "wb") as f:
            f.write(response.content)

        # Return the public URL path (relative to /public)
        return f"demo-images/{filename}"

    except Exception as e:
        print(f"  Image download error: {img_url} — {e}")
        return None


def extract_page_content(soup):
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    title = clean_text(soup.title.get_text()) if soup.title else ""
    text = clean_text(soup.get_text(separator=" "))

    return {"title": title, "full_text": text}


# --------------------------------------------
# Main scraper loop
# --------------------------------------------

all_demos = []

for i in range(START, END + 1):

    demo_id = f"demo{i:03}"
    url = f"{BASE_URL}{demo_id}.htm"

    print(f"Processing: {url}")

    try:
        response = requests.get(url, headers=HEADERS, timeout=20)

        if response.status_code != 200:
            print(f"  Skipping {demo_id} (status {response.status_code})")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        content = extract_page_content(soup)

        images = []
        for idx, img in enumerate(soup.find_all("img")):
            src = img.get("src")
            if not src:
                continue

            img_url = urljoin(url, src)
            filename = safe_filename(img_url) or "image.jpg"
            filename = f"{demo_id}-{idx}-{filename}"
            local_path = download_image(img_url, IMAGE_DIR, filename=filename)

            if local_path:
                images.append({
                    "original_url": img_url,
                    "local_path": local_path,
                    "alt": img.get("alt", "")
                })

        all_demos.append({
            "id": i,
            "demo_id": demo_id,
            "source_url": url,
            "title": content["title"],
            "full_text": content["full_text"],
            "images": images,
        })

        print(f"  Collected: {demo_id} ({len(images)} image(s))")
        time.sleep(0.5)

    except Exception as e:
        print(f"  Error processing {demo_id}: {e}")

# --------------------------------------------
# Write output
# --------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump({"demos": all_demos}, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(all_demos)} demos → {OUTPUT_FILE}")
print(f"Images → {IMAGE_DIR}")
